# 📝 Tarea: Limpieza de Datos - Dirty Cafe Sales

Este Notebook está diseñado para practicar técnicas básicas de limpieza de datos usando la librería **Pandas**. Sigue las instrucciones de cada fase para transformar un conjunto de datos "sucio" en uno listo para el análisis.

### ⚙️ Importación de Librerías
Comienza importando las librerías necesarias (`pandas` y `numpy`).

**NumPy: Numerical Python**
Es la base matemática. Imagínalo como una calculadora súper rápida y potente diseñada específicamente para trabajar con conjuntos enormes de números.

* Sirve para hacer cálculos matemáticos, estadísticos y científicos a gran velocidad.

* Es parecido a una lista tradicional de Python, pero está optimizado para que la computadora lo procese en milisegundos, incluso si contiene millones de datos.

**Pandas:** 🐼
Es la herramienta de organización y análisis. Está construida sobre NumPy, lo que significa que aprovecha toda su velocidad matemática, pero le añade una capa visual y práctica. Imagínalo como un "Excel con superpoderes" que controlas mediante código.

* Sirve para leer, limpiar, filtrar, agrupar y transformar datos.

* Es literalmente una tabla con filas y columnas. A diferencia de NumPy (que solo entiende números y posiciones matemáticas), Pandas entiende etiquetas humanas como "Nombre del Producto", "Fecha" o "Total Gastado".

In [ ]:
import pandas as pd
import numpy as np

---
### 🔍 Fase 1: Conocer los datos
1. Carga el archivo `dirty_cafe_sales.csv`.
2. Muestra las primeras 5 filas.
3. Cuenta cuántos valores nulos tiene cada columna.

> 💡 **Pistas:** `pd.read_csv()`, `df.head()`, `df.isnull().sum()`

In [ ]:
# --- FASE 1: Conocer los datos ---

# Cargamos el archivo CSV en un DataFrame de Pandas.
# Un DataFrame es como una tabla de Excel dentro de Python.
df = pd.read_csv('dirty_cafe_sales.csv')

# Mostramos las primeras 5 filas para ver como lucen los datos.
print('=== Primeras 5 filas ===')
print(df.head())

# Contamos cuantos valores nulos (vacios) tiene cada columna.
# Esto nos ayuda a saber donde hay huecos en la informacion.
print('\n=== Valores nulos por columna ===')
print(df.isnull().sum())


=== Primeras 5 filas ===
  Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2            2.0         4.0     Credit Card   
1    TXN_4977031    Cake        4            3.0        12.0            Cash   
2    TXN_4271903  Cookie        4            1.0       ERROR     Credit Card   
3    TXN_7034554   Salad        2            5.0        10.0         UNKNOWN   
4    TXN_3160411  Coffee        2            2.0         4.0  Digital Wallet   

   Location Transaction Date  
0  Takeaway       2023-09-08  
1  In-store       2023-05-16  
2  In-store       2023-07-19  
3   UNKNOWN       2023-04-27  
4  In-store       2023-06-11  

=== Valores nulos por columna ===
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


---
### 🗑️ Fase 2: Borrar la "Basura" (Textos de error)
El sistema registró errores como las palabras **"ERROR"** y **"UNKNOWN"**. Cámbialas por nulos reales.

1. Reemplaza estas palabras por `np.nan` en todo el DataFrame.

> 💡 **Pistas:** `df.replace(['TEXTO1', 'TEXTO2'], np.nan, inplace=True)`

In [ ]:
# --- FASE 2: Borrar la Basura (textos de error) ---

# El dataset tiene celdas con los textos 'ERROR' y 'UNKNOWN'.
# Estos no son datos reales, son mensajes de error del sistema.
# Los reemplazamos por np.nan (el valor nulo real de NumPy)
# para que Pandas los reconozca como dato faltante.
df.replace(['ERROR', 'UNKNOWN'], np.nan, inplace=True)

# inplace=True significa que el cambio se aplica SOBRE el mismo DataFrame
# en lugar de crear uno nuevo.

# Verificamos cuantos nulos hay ahora (deberan ser mas que antes).
print('=== Nulos despues de limpiar ERROR y UNKNOWN ===')
print(df.isnull().sum())


=== Nulos despues de limpiar ERROR y UNKNOWN ===
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


---
### 🛠️ Fase 3: Arreglar los tipos de datos
Convierte las columnas a sus formatos correctos para poder hacer cálculos.

1. Convierte `Quantity`, `Price Per Unit` y `Total Spent` a numérico.
2. Convierte `Transaction Date` a tipo fecha.

> 💡 **Pistas:** `pd.to_numeric()`, `pd.to_datetime()`

In [ ]:
# --- FASE 3: Arreglar los tipos de datos ---

# Pandas a veces lee las columnas numericas como texto (string),
# especialmente si tenian errores mezclados. Debemos convertirlas.

# pd.to_numeric() convierte una columna a numero.
# errors='coerce' hace que si encuentra algo que no puede convertir,
# lo convierta automaticamente en NaN en lugar de lanzar un error.
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')

# pd.to_datetime() convierte el texto a tipo fecha real.
# Esto permite hacer operaciones con fechas (ordenar, filtrar por mes, etc.).
# errors='coerce' convierte fechas invalidas en NaT (Not a Time = fecha nula).
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

# Mostramos los tipos de datos de cada columna para confirmar el cambio.
print('=== Tipos de datos actuales ===')
print(df.dtypes)


=== Tipos de datos actuales ===
Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object


---
### 🩹 Fase 4: Manejar valores nulos
Vamos a rellenar los huecos o eliminar lo que no sirva.

1. Si falta el `Item`, rellénalo con "Desconocido".
2. Si falta `Payment Method` o `Location`, rellénalos con "No Especificado".
3. Elimina las filas donde `Quantity` o `Price Per Unit` sean nulos.

> 💡 **Pistas:** `df['col'].fillna()`, `df.dropna(subset=['col1', 'col2'])`

In [ ]:
# --- FASE 4: Manejar valores nulos ---

# Estrategia 1: Rellenar nulos con un valor por defecto.
# Si no sabemos que producto fue, escribimos 'Desconocido'.
df['Item'] = df['Item'].fillna('Desconocido')

# Lo mismo para el metodo de pago y la ubicacion.
df['Payment Method'] = df['Payment Method'].fillna('No Especificado')
df['Location'] = df['Location'].fillna('No Especificado')

# Estrategia 2: Eliminar filas donde faltan datos criticos.
# Si no sabemos la cantidad NI el precio, no podemos calcular nada util.
# dropna(subset=[...]) elimina SOLO las filas donde alguna de esas
# columnas sea nula, manteniendo el resto del DataFrame intacto.
df.dropna(subset=['Quantity', 'Price Per Unit'], inplace=True)

# Mostramos un resumen final del DataFrame limpio.
print(f'=== Filas restantes: {len(df)} ===')
print(df.isnull().sum())


=== Filas restantes: 9006 ===
Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent         462
Payment Method        0
Location              0
Transaction Date    410
dtype: int64


---
### 💾 Fase 5: Guardar el resultado
Exporta tu trabajo limpio.

1. Guarda el archivo como `ventas_limpias.csv` sin el índice.

> 💡 **Pistas:** `df.to_csv(index=False)`

In [ ]:
# --- FASE 5: Guardar el resultado ---

# Exportamos el DataFrame limpio a un nuevo archivo CSV.
# index=False evita que Pandas agregue una columna extra con los numeros
# de fila (0, 1, 2...) al archivo guardado, ya que no la necesitamos.
df.to_csv('ventas_limpias.csv', index=False)

print('Archivo guardado correctamente como ventas_limpias.csv')
print(f'El dataset limpio tiene {len(df)} filas y {len(df.columns)} columnas.')


Archivo guardado correctamente como ventas_limpias.csv
El dataset limpio tiene 9006 filas y 8 columnas.


En el mundo del análisis existe una regla de oro llamada "Basura entra, basura sale" (Garbage In, Garbage Out). La limpieza de datos sirve fundamentalmente para tres cosas:

* Evitar decisiones equivocadas: Un punto decimal mal puesto o un dato duplicado pueden alterar por completo los totales, llevando a una empresa a tomar malas decisiones basadas en mentiras numéricas.

* Permitir el análisis automático: Los programas, los gráficos y los modelos de Inteligencia Artificial necesitan reglas estrictas. Si intentas sumar la palabra "ERROR" con un número, el sistema colapsará.

* Generar confianza: Un conjunto de datos limpio te da la seguridad de que tus descubrimientos, porcentajes y conclusiones representan exactamente la realidad de lo que estás midiendo.

**Limpiar los datos es preparar los cimientos para construir un análisis sólido y confiable.**